In [7]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn tqdm requests joblib

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.6/72.0 MB 10.1 MB/s eta 0:00:07
   - -------------------------------------- 3.4/72.0 MB 10.6 MB/s eta 0:00:07
   - -------------------------------------- 3.4/72.0 MB 10.6 MB/s eta 0:00:07
   - -------------------------------------- 3.4/72.0 MB 10.6 MB/s eta 0:00:07
   - -------------------------------------- 3.4/72.0 MB 10.6 MB/s eta 0:00:07
   -- ------------------------------------- 3.7/72.0 MB 2.6 MB/s eta 0:00:26
   -- ------------------------------------- 4.7/72.0 MB 3.0 MB/s eta 0:00:23
   ---- ----------------------------------- 8.1/72.0 MB 4.5 MB/s eta 0:00:15
   ------ --------------------------------- 11.0/72.0 MB 5.5 MB/s eta 0:00:11
   ------ --------------------------------- 11.0/72.0 MB 5.5 MB/s eta 0:00:11
   ------ --------------------------------- 11.0/72.0 MB 5.5 MB/s eta 0:00:11
   --

In [18]:
# 3_test_research_design.ipynb
import os
# ★★★ 強制設定 CPU 核心數，避免 Windows WinError 2 錯誤 ★★★
os.environ['LOKY_MAX_CPU_COUNT'] = '4' 

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- 設定路徑 ---
BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "data_processed.pkl")
RESULT_DIR = os.path.join(BASE_DIR, "results")

# 1. 載入 Research 數據
print("載入數據 (Research Data)...")
data = joblib.load(DATA_PATH)
X, y, fyear = data["X_research"], data["y"], data["fyear"]

# 處理特徵名稱鍵值不一致的保險機制
if "feat_names" in data:
    feat_names = data["feat_names"]
elif "feature_names_research" in data:
    feat_names = data["feature_names_research"]
else:
    feat_names = [f"Var_{i}" for i in range(X.shape[1])]

print(f"使用特徵: {len(feat_names)} 個 (Full Features)")

# 2. 切分
train_mask = (fyear >= 2019) & (fyear <= 2023)
test_mask = (fyear == 2024)
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# ---------------------------------------------------------
# 3. 定義 5 種模型 (The 5 Configurations)
# ---------------------------------------------------------
# Sklearn 使用 C 作為正則化強度的倒數 (C = 1/lambda)
# lambda=10  => C=0.1
# lambda=100 => C=0.01

models_config = {
    # 1. 單純 Logistic Regression (幾乎無懲罰)
    "Simple LR": LogisticRegression(penalty='l2', C=1e9, solver='lbfgs', random_state=42),
    
    # 2. Ridge (L2) lambda=10
    "Ridge (λ=10)": LogisticRegression(penalty='l2', C=0.1, solver='liblinear', random_state=42),
    
    # 3. Ridge (L2) lambda=100
    "Ridge (λ=100)": LogisticRegression(penalty='l2', C=0.01, solver='liblinear', random_state=42),
    
    # 4. Lasso (L1) lambda=10 
    "Lasso (λ=10)": LogisticRegression(penalty='l1', C=0.1, solver='liblinear', random_state=42),
    
    # 5. Lasso (L1) lambda=100 
    "Lasso (λ=100)": LogisticRegression(penalty='l1', C=0.01, solver='liblinear', random_state=42)
}

# ---------------------------------------------------------
# 4. 定義 2 種採樣策略
# ---------------------------------------------------------
sampling_strategies = {
    "No SMOTE": None,
    "SMOTE": SMOTE(random_state=42)
}

# ---------------------------------------------------------
# 5. 執行 2x5 實驗矩陣 (共 10 組)
# ---------------------------------------------------------
results_list = []
lasso_coefs = {} 

print(f"\n開始執行 2x5 實驗矩陣 (共 {len(sampling_strategies) * len(models_config)} 組)...")

for smote_name, sampler in sampling_strategies.items():
    for model_name, model in models_config.items():
        
        full_method_name = f"{smote_name} + {model_name}"
        print(f"Running: {full_method_name}...")
        
        # 建立 Pipeline
        steps = []
        if sampler: 
            steps.append(('sampler', sampler))
        steps.append(('clf', model))
        
        pipeline = ImbPipeline(steps)
        
        # 訓練
        pipeline.fit(X_train, y_train)
        
        # 預測
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        # 評估
        auc_score = roc_auc_score(y_test, y_prob)
        brier = brier_score_loss(y_test, y_prob)
        
        # 存結果
        results_list.append({
            "Method": full_method_name,
            "Sampling": smote_name,
            "Model": model_name,
            "AUC": auc_score,
            "Brier": brier
        })
        
        # 儲存 Lasso (No SMOTE) 的係數，供後續畫圖分析
        if "Lasso" in model_name and smote_name == "No SMOTE":
            lasso_coefs[model_name] = pipeline.named_steps['clf'].coef_[0]

# ---------------------------------------------------------
# 6. 存檔
# ---------------------------------------------------------
df_res = pd.DataFrame(results_list)

print("\n[實驗結果摘要 - 依照 AUC 排序]")
print(df_res.sort_values(by="AUC", ascending=False))

joblib.dump({
    "results": df_res, 
    "lasso_coefs": lasso_coefs, 
    "feats": feat_names
}, os.path.join(RESULT_DIR, "research_results.pkl"))

print(f"\nResearch Experiment Complete. 結果已儲存。")

載入數據 (Research Data)...
使用特徵: 22 個 (Full Features)

開始執行 2x5 實驗矩陣 (共 10 組)...
Running: No SMOTE + Simple LR...
Running: No SMOTE + Ridge (λ=10)...
Running: No SMOTE + Ridge (λ=100)...
Running: No SMOTE + Lasso (λ=10)...
Running: No SMOTE + Lasso (λ=100)...
Running: SMOTE + Simple LR...
Running: SMOTE + Ridge (λ=10)...
Running: SMOTE + Ridge (λ=100)...
Running: SMOTE + Lasso (λ=10)...
Running: SMOTE + Lasso (λ=100)...

[實驗結果摘要 - 依照 AUC 排序]
                     Method  Sampling          Model       AUC     Brier
2  No SMOTE + Ridge (λ=100)  No SMOTE  Ridge (λ=100)  0.722870  0.036076
7     SMOTE + Ridge (λ=100)     SMOTE  Ridge (λ=100)  0.720951  0.263570
9     SMOTE + Lasso (λ=100)     SMOTE  Lasso (λ=100)  0.720267  0.264893
6      SMOTE + Ridge (λ=10)     SMOTE   Ridge (λ=10)  0.720249  0.264035
8      SMOTE + Lasso (λ=10)     SMOTE   Lasso (λ=10)  0.720162  0.264259
3   No SMOTE + Lasso (λ=10)  No SMOTE   Lasso (λ=10)  0.720033  0.035937
5         SMOTE + Simple LR     SMOTE      Sim